# YouTube Data Ingestion for SpeechBrain SSL Training

This notebook downloads audio from YouTube links, preprocesses it (converts to 16kHz mono WAV, segments), and generates a manifest file suitable for the `speechbrain_ssl_training_colab.ipynb` notebook.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -r requirements.txt
print("--- Dependencies Installed ---")

# Ensure ffmpeg is available (usually it is on Colab)
import subprocess
try:
    subprocess.run(['ffmpeg', '-version'], capture_output=True, check=True)
    print("ffmpeg is available.")
except FileNotFoundError:
    print("ffmpeg not found. Installing...")
    subprocess.run(['apt-get', 'update'], capture_output=True, check=True)
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True, check=True)
    print("ffmpeg installed.")
except subprocess.CalledProcessError:
    print("ffmpeg found, but there was an error running it. This might be an issue.")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("--- Google Drive Mounted ---")

## 2. User Input: YouTube Links

You have two options to provide the YouTube links:
1.  **Paste Links Directly**: Set `INPUT_METHOD = "PASTE"` and paste your YouTube links into the `youtube_links_text_area` field that appears when you run the next cell.
2.  **Use a File from Google Drive**: Set `INPUT_METHOD = "FILE"` and provide the path to your `youtube_links.txt` file on Google Drive in the `YOUTUBE_LINKS_FILE_DRIVE_PATH` variable.

In [ ]:
import os

#@title { vertical-output: true }
INPUT_METHOD = "PASTE" #@param ["PASTE", "FILE"]
YOUTUBE_LINKS_FILE_DRIVE_PATH = "/content/drive/MyDrive/your_project_repo/youtube_links.txt" #@param {type:"string"}
youtube_links_text_area = """#@param {type:"string"}
# Paste YouTube links here, one per line, if INPUT_METHOD is 'PASTE'
https://www.youtube.com/watch?v=example1
https://www.youtube.com/watch?v=example2
"""

COLAB_TEMP_LINKS_FILE = "youtube_links_colab.txt"
final_links_file_to_use = None

if INPUT_METHOD == "PASTE":
    with open(COLAB_TEMP_LINKS_FILE, 'w') as f:
        f.write(youtube_links_text_area)
    final_links_file_to_use = COLAB_TEMP_LINKS_FILE
    print(f"Using pasted links. Saved to temporary file: {os.path.abspath(COLAB_TEMP_LINKS_FILE)}")
elif INPUT_METHOD == "FILE":
    if not os.path.exists(YOUTUBE_LINKS_FILE_DRIVE_PATH):
        raise FileNotFoundError(f"YouTube links file not found on Google Drive: {YOUTUBE_LINKS_FILE_DRIVE_PATH}")
    final_links_file_to_use = YOUTUBE_LINKS_FILE_DRIVE_PATH
    print(f"Using links file from Google Drive: {final_links_file_to_use}")
else:
    raise ValueError("Invalid INPUT_METHOD. Choose 'PASTE' or 'FILE'.")

if final_links_file_to_use:
    with open(final_links_file_to_use, 'r') as f:
        link_count = len([line for line in f if line.strip() and not line.startswith('#')])
    print(f"Found {link_count} valid links to process.")

## 3. Configuration: Output Paths on Google Drive

Define where the downloaded and processed audio data, and the final manifest, will be stored on your Google Drive.

In [ ]:
import os

# --- User Configuration for Drive Paths ---
BASE_DRIVE_PATH = "/content/drive/MyDrive/your_project_repo_on_drive/youtube_ssl_data" # TODO: Change this to your desired base project folder on Drive
# --- End User Configuration ---

RAW_AUDIO_DOWNLOAD_DIR = os.path.join(BASE_DRIVE_PATH, "0_raw_youtube_audio")
CONVERTED_AUDIO_DIR = os.path.join(BASE_DRIVE_PATH, "1_converted_16khz_mono")
SEGMENTED_AUDIO_DIR = os.path.join(BASE_DRIVE_PATH, "2_segmented_audio")
MANIFEST_DIR = os.path.join(BASE_DRIVE_PATH, "3_manifests")
FINAL_MANIFEST_FILE = os.path.join(MANIFEST_DIR, "youtube_audio_manifest.json")

print(f"Base output path on Drive: {BASE_DRIVE_PATH}")

paths_to_create = [RAW_AUDIO_DOWNLOAD_DIR, CONVERTED_AUDIO_DIR, SEGMENTED_AUDIO_DIR, MANIFEST_DIR]
for p in paths_to_create:
    os.makedirs(p, exist_ok=True)
    print(f"Ensured directory exists: {p}")

print("--- Output paths configured ---")

## 4. Download YouTube Audio

This cell adapts the logic from `runyoro_speech_ai/data_ingestion/download_youtube.py`.

In [ ]:
import subprocess
import logging
import os

logger_dl = logging.getLogger("youtube_downloader_colab")
if not logger_dl.hasHandlers():
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger_dl.addHandler(handler)
logger_dl.setLevel(logging.INFO)

def download_single_video_colab(video_url, output_dir):
    logger_dl.info(f"Attempting to download: {video_url} into {output_dir}")
    output_template = os.path.join(output_dir, '%(id)s.%(ext)s')
    command = ['yt-dlp', '-x', '--audio-format', 'best', '-o', output_template, '--no-playlist', '--embed-metadata', video_url]
    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True, encoding='utf-8')
        logger_dl.info(f"Successfully downloaded: {video_url}")
        return True
    except subprocess.CalledProcessError as e:
        logger_dl.error(f"Error downloading {video_url}: {e.stderr.strip()}")
        return False
    except FileNotFoundError:
        logger_dl.error("yt-dlp not found. Ensure it's installed (should be from requirements.txt).")
        raise
    return False

if not final_links_file_to_use or not os.path.exists(final_links_file_to_use):
    raise ValueError("final_links_file_to_use is not set or file does not exist. Run Cell 2 first.")

with open(final_links_file_to_use, 'r') as f:
    urls_to_process = [line.strip() for line in f if line.strip() and not line.startswith('#')]

logger_dl.info(f"Starting download process for {len(urls_to_process)} URLs. Output directory: {RAW_AUDIO_DOWNLOAD_DIR}")
overall_success_dl = True
for i, url in enumerate(urls_to_process):
    logger_dl.info(f"Processing URL {i+1}/{len(urls_to_process)}: {url}")
    if not download_single_video_colab(url, RAW_AUDIO_DOWNLOAD_DIR):
        overall_success_dl = False

if overall_success_dl:
    logger_dl.info("All YouTube downloads attempted successfully (or files already existed in target format).")
else:
    logger_dl.warning("Some YouTube downloads failed. Check logs above.")
print("--- YouTube Download Stage Complete ---")

## 5. Preprocess Audio - Part 1: Conversion to 16kHz Mono WAV

This cell adapts logic from `runyoro_speech_ai/data_ingestion/preprocess_audio.py` to convert downloaded audio to a standard format.

In [ ]:
from pydub import AudioSegment
from pydub.exceptions import CouldntDecodeError
import os
import logging

logger_conv = logging.getLogger("audio_converter_colab")
if not logger_conv.hasHandlers():
    handler_conv = logging.StreamHandler()
    formatter_conv = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler_conv.setFormatter(formatter_conv)
    logger_conv.addHandler(handler_conv)
logger_conv.setLevel(logging.INFO)

TARGET_SAMPLING_RATE_HZ = 16000
SUPPORTED_EXTENSIONS_CONV = ['.wav', '.mp3', '.m4a', '.ogg', '.flac', '.opus', '.mp4', '.mkv', '.mov', '.avi', '.webm', '.flv', '.wmv']

def convert_to_standard_wav_colab(filepath, output_dir):
    filename = os.path.basename(filepath)
    basename, _ = os.path.splitext(filename)
    output_filename = f"{basename}.wav"
    output_filepath = os.path.join(output_dir, output_filename)

    if os.path.exists(output_filepath):
        logger_conv.info(f"Converted file {output_filepath} already exists. Skipping.")
        return output_filepath
    try:
        logger_conv.info(f"Converting: {filepath}")
        audio = AudioSegment.from_file(filepath)
        audio = audio.set_channels(1)
        audio = audio.set_frame_rate(TARGET_SAMPLING_RATE_HZ)
        audio.export(output_filepath, format="wav")
        logger_conv.info(f"  Successfully converted to {output_filepath}")
        return output_filepath
    except CouldntDecodeError:
        logger_conv.error(f"Could not decode {filepath}. Corrupted or unsupported format.")
    except FileNotFoundError: 
        logger_conv.error(f"ffmpeg/ffprobe not found for {filepath}. Ensure ffmpeg is installed.")
        raise 
    except Exception as e:
        logger_conv.error(f"Error converting {filepath}: {e}")
    return None

logger_conv.info(f"--- Starting Audio Conversion Stage ---")
logger_conv.info(f"Input directory for conversion: {RAW_AUDIO_DOWNLOAD_DIR}")
logger_conv.info(f"Converted 16kHz mono WAVs will be saved to: {CONVERTED_AUDIO_DIR}")

converted_files_count = 0
error_files_count_conv = 0
for root, _, files in os.walk(RAW_AUDIO_DOWNLOAD_DIR):
    for file in files:
        filepath = os.path.join(root, file)
        _, ext = os.path.splitext(file)
        if ext.lower() in SUPPORTED_EXTENSIONS_CONV:
            if convert_to_standard_wav_colab(filepath, CONVERTED_AUDIO_DIR):
                converted_files_count += 1
            else:
                error_files_count_conv += 1
        else:
            logger_conv.debug(f"Skipping {file}, not a supported media extension.")

logger_conv.info(f"Conversion Stage Complete. Converted: {converted_files_count}, Failed: {error_files_count_conv}")
print("--- Audio Conversion Stage Complete ---")

## 6. Preprocess Audio - Part 2: Segmentation

This cell adapts logic from `runyoro_speech_ai/data_ingestion/preprocess_audio.py` to segment the converted WAV files.

In [ ]:
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os
import logging

logger_seg = logging.getLogger("audio_segmenter_colab")
if not logger_seg.hasHandlers():
    handler_seg = logging.StreamHandler()
    formatter_seg = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler_seg.setFormatter(formatter_seg)
    logger_seg.addHandler(handler_seg)
logger_seg.setLevel(logging.INFO)

# --- Segmentation Parameters (User Adjustable) ---
MIN_SILENCE_LEN_MS = 700        #@param {type:"integer"}
SILENCE_THRESH_DBFS = -45       #@param {type:"integer"}
KEEP_SILENCE_MS = 250           #@param {type:"integer"}
MIN_SEGMENT_DURATION_MS = 2000  #@param {type:"integer"}
MAX_SEGMENT_DURATION_MS = 20000 #@param {type:"integer"}
TARGET_SPLIT_DURATION_MS = 10000#@param {type:"integer"}
# --- End Segmentation Parameters ---

def force_split_segment_colab(audio_segment, target_duration_ms, min_split_duration_ms, base_output_path, original_filename_prefix):
    num_chunks = 0; duration = len(audio_segment); start_time = 0; segment_counter = 0
    while start_time < duration:
        end_time = start_time + target_duration_ms; current_chunk_duration = end_time - start_time
        if end_time > duration: current_chunk_duration = duration - start_time; end_time = duration
        if current_chunk_duration < min_split_duration_ms:
            if segment_counter > 0: logger_seg.info(f"    Force split: Last chunk too small ({current_chunk_duration}ms), discarding."); break
        chunk = audio_segment[start_time:end_time]
        segment_filename = f"{original_filename_prefix}_fs_{segment_counter:03d}.wav"
        try:
            chunk.export(os.path.join(base_output_path, segment_filename), format="wav")
            logger_seg.info(f"    Force split: Saved chunk {segment_filename} ({len(chunk)}ms)")
            num_chunks += 1
        except Exception as e: logger_seg.error(f"    Force split error: {e}")
        segment_counter += 1; start_time = end_time
        if start_time >= duration: break
    return num_chunks

def segment_wav_file_colab(wav_filepath, output_dir, min_silence_len, silence_thresh, keep_silence, min_duration_ms, max_duration_ms, target_split_duration_ms):
    original_fn_no_ext = os.path.splitext(os.path.basename(wav_filepath))[0]
    logger_seg.info(f"Segmenting: {wav_filepath}")
    try: audio = AudioSegment.from_wav(wav_filepath)
    except Exception as e: logger_seg.error(f"  Error loading {wav_filepath}: {e}"); return 0
    segments = split_on_silence(audio, min_silence_len, silence_thresh, keep_silence, seek_step=1)
    total_saved = 0
    if not segments:
        logger_seg.info(f"  No segments from silence detection for {original_fn_no_ext}.")
        audio_len = len(audio)
        if audio_len > max_duration_ms: total_saved += force_split_segment_colab(audio, target_split_duration_ms, min_duration_ms, output_dir, original_fn_no_ext)
        elif audio_len >= min_duration_ms: 
            try: audio.export(os.path.join(output_dir, f"{original_fn_no_ext}_full.wav"), format="wav"); total_saved += 1; logger_seg.info(f"  Saved full audio as one segment for {original_fn_no_ext}")
            except Exception as e: logger_seg.error(f"  Error exporting full audio for {original_fn_no_ext}: {e}")
        else: logger_seg.info(f"  Full audio duration ({audio_len}ms) < min_duration_ms. Discarding {original_fn_no_ext}.")
        return total_saved
    for i, segment in enumerate(segments):
        seg_len = len(segment)
        if seg_len < min_duration_ms: logger_seg.info(f"    Segment {i} of {original_fn_no_ext} too short ({seg_len}ms). Discarded."); continue
        if seg_len > max_duration_ms:
            total_saved += force_split_segment_colab(segment, target_split_duration_ms, min_duration_ms, output_dir, f"{original_fn_no_ext}_seg_{i:03d}")
        else:
            try: segment.export(os.path.join(output_dir, f"{original_fn_no_ext}_seg_{i:03d}.wav"), format="wav"); total_saved +=1; logger_seg.info(f"    Saved segment {i} for {original_fn_no_ext} ({seg_len}ms).")
            except Exception as e: logger_seg.error(f"    Error exporting segment {i} for {original_fn_no_ext}: {e}")
    return total_saved

logger_seg.info(f"--- Starting Audio Segmentation Stage ---")
logger_seg.info(f"Input directory for segmentation: {CONVERTED_AUDIO_DIR}")
logger_seg.info(f"Segmented audio clips will be saved to: {SEGMENTED_AUDIO_DIR}")
total_segments_overall = 0; files_processed_seg = 0
for item in os.listdir(CONVERTED_AUDIO_DIR):
    if item.lower().endswith(".wav"):
        filepath = os.path.join(CONVERTED_AUDIO_DIR, item)
        if os.path.isfile(filepath):
            files_processed_seg +=1
            total_segments_overall += segment_wav_file_colab(filepath, SEGMENTED_AUDIO_DIR, MIN_SILENCE_LEN_MS, SILENCE_THRESH_DBFS, KEEP_SILENCE_MS, MIN_SEGMENT_DURATION_MS, MAX_SEGMENT_DURATION_MS, TARGET_SPLIT_DURATION_MS)
logger_seg.info(f"Segmentation Stage Complete. Processed {files_processed_seg} files. Total segments generated: {total_segments_overall}")
print("--- Audio Segmentation Stage Complete ---")

## 7. Generate Manifest File

This cell adapts logic from `runyoro_speech_ai/scripts/generate_manifest.py` to create a JSON manifest file compatible with SpeechBrain (keys as utterance IDs).

In [ ]:
import json
from pydub import AudioSegment
import os
import logging

logger_manifest = logging.getLogger("manifest_generator_colab")
if not logger_manifest.hasHandlers():
    handler_manifest = logging.StreamHandler()
    formatter_manifest = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler_manifest.setFormatter(formatter_manifest)
    logger_manifest.addHandler(handler_manifest)
logger_manifest.setLevel(logging.INFO)

logger_manifest.info(f"--- Starting Manifest Generation ---")
logger_manifest.info(f"Input directory for manifest (segmented audio): {SEGMENTED_AUDIO_DIR}")
logger_manifest.info(f"Output manifest file: {FINAL_MANIFEST_FILE}")

manifest_data = {}
files_processed_manifest = 0
files_error_manifest = 0

for root, _, files in os.walk(SEGMENTED_AUDIO_DIR):
    for file in files:
        if file.lower().endswith(".wav"):
            filepath = os.path.join(root, file)
            abs_filepath = os.path.abspath(filepath) # Ensure absolute paths
            utt_id = os.path.splitext(file)[0] # Use filename without extension as utterance ID
            try:
                audio = AudioSegment.from_wav(abs_filepath)
                duration_seconds = round(audio.duration_seconds, 3)
                manifest_data[utt_id] = {
                    "wav": abs_filepath, # SpeechBrain often uses 'wav' key for path
                    "duration": duration_seconds
                    # Add other fields here if your SpeechBrain hparams expect them for SSL targets,
                    # e.g., speaker_id, although for SSL with k-means this might be minimal.
                }
                files_processed_manifest += 1
                logger_manifest.debug(f"Added to manifest: {utt_id} -> {abs_filepath}, Duration: {duration_seconds}s")
            except Exception as e:
                logger_manifest.warning(f"Error processing {abs_filepath} for manifest: {e}. Skipping.")
                files_error_manifest += 1

if not manifest_data:
    logger_manifest.warning("No WAV files found or processed in segmentation directory. Manifest will be empty.")
else:
    try:
        with open(FINAL_MANIFEST_FILE, 'w', encoding='utf-8') as f_out:
            json.dump(manifest_data, f_out, indent=2, ensure_ascii=False)
        logger_manifest.info(f"Manifest generation complete. Processed: {files_processed_manifest}, Errors: {files_error_manifest}")
        logger_manifest.info(f"Manifest saved to: {FINAL_MANIFEST_FILE}")
    except IOError as e:
        logger_manifest.error(f"Could not write manifest to {FINAL_MANIFEST_FILE}: {e}")

print("--- Manifest Generation Complete ---")

## 8. Next Steps

Data ingestion and preprocessing are complete!

1.  **Verify Outputs**: Check the directories on your Google Drive:
    *   `BASE_DRIVE_PATH/0_raw_youtube_audio/`: Should contain downloaded audio from YouTube.
    *   `BASE_DRIVE_PATH/1_converted_16khz_mono/`: Should contain WAV files, converted to 16kHz mono.
    *   `BASE_DRIVE_PATH/2_segmented_audio/`: Should contain the final segmented audio clips.
    *   `BASE_DRIVE_PATH/3_manifests/youtube_audio_manifest.json`: This is the manifest file you'll use.
2.  **K-Means Targets**: The `speechbrain_ssl_training_colab.ipynb` expects k-means targets (`<utt_id>_kmeans_labels.npy`) to be generated and placed in a directory specified by `hparams.target_label_dir`. You will need to run the k-means generation script (`runyoro_speech_ai/speechbrain_ssl_training/generate_kmeans_targets.py`) using the manifest and audio from this notebook as input, and ensure its output directory matches what the SSL training notebook expects. This step is NOT covered by this ingestion notebook.
3.  **Update SpeechBrain SSL Training Notebook**: 
    *   Open `notebooks/speechbrain_ssl_training_colab.ipynb`.
    *   In its **Configuration Cell** (Cell 6):
        *   Set `DATA_FOLDER_DRIVE` to the parent directory of your segmented audio files if your manifest uses relative paths, OR ensure paths in `youtube_audio_manifest.json` are absolute (this notebook generates absolute paths).
        *   Set `TRAIN_MANIFEST_FULL_PATH_DRIVE` to point to `BASE_DRIVE_PATH/3_manifests/youtube_audio_manifest.json` (i.e., the `FINAL_MANIFEST_FILE` variable from this notebook).
        *   Ensure `KMEANS_TARGET_DIR_FULL_PATH_DRIVE` in the SSL training notebook points to where your k-means `.npy` files are stored for these utterances.
    *   The `hparams_ssl.yaml` used by the SSL training notebook will be updated to reflect these Drive paths.
4.  **Run SSL Training**: Proceed with running the `speechbrain_ssl_training_colab.ipynb`.